In [29]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

import numpy as np

from sklearn.model_selection import train_test_split 
import os

In [6]:
DEVICE  = torch.device(f"cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE  = "cpu"
print(DEVICE)

cpu


In [36]:
base_path = "../Dataset"

dataset = base_path + "/s2-utm-33N-18E-242N-2018"
train_geojson_path = base_path + "/br-18E-242N-crop-labels-train-2018.geojson"

folder = "/DS3"
subfolder1 = "/Temporal Data"

data_path = dataset+folder
temporal_data_path = dataset+folder+subfolder1

classification_model_path = "../Classificaton models/Spatial"
temporal_classification_model_path = "../Classificaton models/Temporal"


In [3]:
data = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/11_final_gan_data_without_mixed_patches_64X64_majority50.npy')
labels = np.load('../Dataset/s2-utm-33N-18E-242N-2018/DS3/12_final_gan_labels_without_mixed_patches_64X64_majority50.npy')

labels_upadated = np.where(labels == -1, 0, labels)
train_data, validation_data, train_labels, validation_labels = train_test_split(data, labels_upadated, random_state= 2) 


In [20]:
class ImageLabelDataset(Dataset):
    def __init__(self, image_array, label_array):
        self.images = torch.tensor(image_array, dtype=torch.float32)  # (N, 4, 64, 64)
        self.labels = torch.tensor(label_array, dtype=torch.float32)  # (N, 1, 64, 64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

image_data = np.transpose(train_data, (0, 3, 1, 2))  # (N, 4, 64, 64)

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
def save_models(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    segmentation_model_name,
    encoder_name,
    path
):
    filename = f'/Classification_Model_Pipeline-{segmentation_model_name}_Encoder-{encoder_name}'+r'-Epoch '+str(epoch)+r'.pth'
    full_path = path + filename

    payload = {
        "epochs": int(epoch),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
    }
    torch.save(payload, full_path)

    print(f"Models saved to {full_path}")

def load_models(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    device="cpu",
):
    """
    Load a training checkpoint. Returns the next epoch index to continue from.
    Loads only what you pass in (model, optimizer, scheduler, scaler).
    """
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch = int(checkpoint.get("epochs"))
    return start_epoch


In [21]:
NUM_CLASSES = 10
EPOCHS = 25


In [33]:

# U-Net with ResNet-150 encoder
unet_r50 = smp.Unet(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# DeepLabV3+ with ResNet-50
deeplab_r50 = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# U-Net with ResNet-101 encoder
unet_r101 = smp.Unet(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

# DeepLabV3+ with ResNet-101
deeplab_r101 = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=4,
    classes=NUM_CLASSES
)

model_list = [unet_r50, deeplab_r50, unet_r101, unet_r101]
model_name_list = ['unet_r50', 'deeplab_r50', 'unet_r101', 'unet_r101']
encoder_name_list = ['resnet50', 'resnet101']


In [ ]:
selected_model_index = 0
model = model_list[selected_model_index]


# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()


Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\2501113715.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 1.0820 | Epoch avg loss: 0.947491348362886
Epoch: 1
Loss: 1.2705 | Epoch avg loss: 0.8142219352034422
Epoch: 2
Loss: 0.9687 | Epoch avg loss: 0.7767942524873294
Epoch: 3
Loss: 0.7167 | Epoch avg loss: 0.7065043655725626
Epoch: 4
Loss: 0.8948 | Epoch avg loss: 0.6936028439265031
Epoch: 5
Loss: 1.8549 | Epoch avg loss: 0.674653604053534
Epoch: 6
Loss: 1.6514 | Epoch avg loss: 0.6357284036393349
Epoch: 7
Loss: 1.3906 | Epoch avg loss: 0.6047303028977834
Epoch: 8
Loss: 0.9513 | Epoch avg loss: 0.6001462096778246
Epoch: 9
Loss: 0.8142 | Epoch avg loss: 0.5683856655198795
Epoch: 10
Loss: 0.7140 | Epoch avg loss: 0.5367651968621291
Epoch: 11
Loss: 0.6080 | Epoch avg loss: 0.5249096613663894
Epoch: 12
Loss: 0.6420 | Epoch avg loss: 0.49786755671867955
Epoch: 13
Loss: 0.8296 | Epoch avg loss: 0.48204440021744144
Epoch: 14
Loss: 0.6087 | Epoch avg loss: 0.4327713855757163
Epoch: 15
Loss: 0.4429 | Epoch avg loss: 0.43541620299220085
Epoch: 16
Loss: 1.1261 | Epoch avg loss: 0.421924630609842

In [ ]:
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'
save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


In [43]:
selected_model_index = 1
model = model_list[selected_model_index]


# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()


Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\1914483387.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):
c:\Users\2405647\.conda\envs\myenv\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Loss: 1.0820 | Epoch avg loss: 1.2058267484490688
Epoch: 1
Loss: 0.7749 | Epoch avg loss: 0.9154312066160716
Epoch: 2
Loss: 1.2721 | Epoch avg loss: 0.8265384189211405
Epoch: 3
Loss: 1.1851 | Epoch avg loss: 0.7798352975111741
Epoch: 4
Loss: 0.8794 | Epoch avg loss: 0.7169282447833282
Epoch: 5
Loss: 1.1168 | Epoch avg loss: 0.7024523673149256
Epoch: 6
Loss: 1.2464 | Epoch avg loss: 0.6715920527394001
Epoch: 7
Loss: 1.4771 | Epoch avg loss: 0.6333245273966056
Epoch: 8
Loss: 1.1352 | Epoch avg loss: 0.6571530261291907
Epoch: 9
Loss: 0.9161 | Epoch avg loss: 0.5820433497428894
Epoch: 10
Loss: 0.5373 | Epoch avg loss: 0.5952036893711641
Epoch: 11
Loss: 0.5776 | Epoch avg loss: 0.5382423432400594
Epoch: 12
Loss: 1.0736 | Epoch avg loss: 0.5294359621520226
Epoch: 13
Loss: 0.7589 | Epoch avg loss: 0.5023206902238039
Epoch: 14
Loss: 1.0502 | Epoch avg loss: 0.5136622482767472
Epoch: 15
Loss: 0.8057 | Epoch avg loss: 0.4578948989510536
Epoch: 16
Loss: 0.5208 | Epoch avg loss: 0.4443558007478714

In [45]:
segmentation_model_name = model_name_list[selected_model_index]
encoder_name = 'resnet50'if selected_model_index<2 else 'resnet101'

save_models(model,
            opt, 
            sched, 
            scaler, 
            (epoch+1), 
            segmentation_model_name, 
            encoder_name, 
            classification_model_path)


In [46]:
selected_model_index = 2
model = model_list[selected_model_index]


# Loss: CE + Dice (multiclass)
dice = smp.losses.DiceLoss(mode="multiclass")
ce   = nn.CrossEntropyLoss()   # add class weights if needed
def loss_fn(logits, y): return 0.5*ce(logits, y) + 0.5*dice(logits, y)

# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

# Mixed precision training loop (sketch)
scaler = torch.amp.GradScaler(DEVICE.type)

for epoch in range(EPOCHS):
    print(f'Epoch: {epoch}')
    epoch_loss = []
    model.train()

    for x, y in loader:       # x:(B,4,64,64) in [-1,1], y:(B,64,64) long
        x, y = x.clone().detach().to(DEVICE), y.clone().detach().long().to(DEVICE)

        with torch.cuda.amp.autocast(DEVICE):
            logits = model(x)
            loss = loss_fn(logits, y)
            epoch_loss.append(loss.item())
        
        
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()

    print(f'Loss: {loss.item():.4f} | Epoch avg loss: {sum(epoch_loss)/len(epoch_loss)}')
    sched.step()


Epoch: 0


C:\Users\2405647\AppData\Local\Temp\ipykernel_25828\3051063026.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(DEVICE):


Loss: 1.4185 | Epoch avg loss: 1.264960691332817
Epoch: 1
Loss: 0.8875 | Epoch avg loss: 0.9624979450152471
Epoch: 2
Loss: 1.4594 | Epoch avg loss: 0.8491801854509574
Epoch: 3
Loss: 1.4002 | Epoch avg loss: 0.7888515786482737
Epoch: 4
Loss: 0.8705 | Epoch avg loss: 0.7324007462996703
Epoch: 5


KeyboardInterrupt: 